In [1]:
from pypdf import PdfReader
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import re
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from google import genai

### Reading the PDF and Extracting the text

In [ ]:
pdf = []
try:
    reader = PdfReader('./src/pdfs/bom.pdf')
    for page in reader.pages:
         pdf.append(page.extract_text())

    print(f"Total Pages: {len(pdf)}")

except Exception as exc:
    # TO Catch error 
    print("Remaining")

Total Pages: 224


### Utility Function 

In [3]:

def normalize_text(text: str) -> str:
    """ 
    Cleans extracted PDF text for better chunking and retrieval. 
    """
    # Step 1: Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # # Step 2: Remove repeated book title/header
    # text = re.sub("Basics of Mathematics\s+\d+\s*", "", text)

    # # Step 3: Remove standalone page numbers
    # text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # Step 4: Replace multiple space with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Step 5: Replace multiple blank lines with at most two
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text


# def compute_line_similarity(pdf, n_lines=3, similarity_fn=fuzz.ratio):
#     """
#     Compare the first and last `n_lines` of consecutive PDF pages.

#     Parameters
#     ----------
#     pdf : list[str]
#         List containing page-wise extracted text.

#     n_lines : int
#         Number of header/footer candidate lines.

#     similarity_fn : function
#         RapidFuzz similarity function.
#         Examples:
#             fuzz.ratio
#             fuzz.partial_ratio
#             fuzz.token_sort_ratio
#             fuzz.token_set_ratio
#             fuzz.WRatio

#     Returns
#     -------
#     list[dict]
#         Similarity information for every consecutive page pair.
#     """

#     results = []

#     for page_no, (current, nxt) in enumerate(zip(pdf, pdf[1:]), start=1):

#         current_lines = [line.strip() for line in current.splitlines() if line.strip()]
#         next_lines = [line.strip() for line in nxt.splitlines() if line.strip()]

#         current_header = current_lines[:n_lines]
#         next_header = next_lines[:n_lines]

#         current_footer = current_lines[-n_lines:]
#         next_footer = next_lines[-n_lines:]

#         page_result = {
#             "pages": (page_no, page_no + 1),
#             "header_scores": [],
#             "footer_scores": []
#         }

#         # -------- Header Comparison --------
#         for idx, (line1, line2) in enumerate(zip(current_header, next_header), start=1):

#             score = similarity_fn(line1, line2)

#             page_result["header_scores"].append({
#                 "line": idx,
#                 "text1": line1,
#                 "text2": line2,
#                 "score": score
#             })

#         # -------- Footer Comparison --------
#         for idx, (line1, line2) in enumerate(zip(current_footer, next_footer), start=1):

#             score = similarity_fn(line1, line2)

#             page_result["footer_scores"].append({
#                 "line": idx,
#                 "text1": line1,
#                 "text2": line2,
#                 "score": score
#             })

#         results.append(page_result)

#     return results


# =========================
# 3. Normalize Page Lines
# =========================

def get_page_lines(page):

    return [
        line.strip()
        for line in page.splitlines()
        if line.strip()
    ]

# =========================
# 4. Compute Line Similarity
# =========================


def compute_line_similarity(pdf, candidate_lines=3):
    """
    Compare the first and last `n_lines` of consecutive PDF pages.

    Parameters
    ----------
    pdf : list[str]
        List containing page-wise extracted text.

    n_lines : int
        Number of header/footer candidate lines.

    similarity_fn : function
        RapidFuzz similarity function.
        Examples:
            fuzz.ratio
            fuzz.partial_ratio
            fuzz.token_sort_ratio
            fuzz.token_set_ratio
            fuzz.WRatio

    Returns
    -------
    list[dict]
        Similarity information for every consecutive page pair.
    """

    similarity_records = []

    for page_number, (current_page, next_page) in enumerate(
        zip(pdf, pdf[1:]),
        start=1
    ):

        current_lines = get_page_lines(current_page)
        next_lines = get_page_lines(next_page)

        # Candidate header lines
        current_header = current_lines[:candidate_lines]
        next_header = next_lines[:candidate_lines]

        # Candidate footer lines
        current_footer = current_lines[-candidate_lines:]
        next_footer = next_lines[-candidate_lines:]

        # Compare header lines
        for line_number, (text1, text2) in enumerate(
            zip(current_header, next_header),
            start=1
        ):

            similarity_records.append({
                "page_from": page_number,
                "page_to": page_number + 1,
                "text1": text1,
                "text2": text2,
                "region": "header",
                "line": line_number,
                "score": fuzz.ratio(text1, text2)
            })

        # Compare footer lines
        for line_number, (text1, text2) in enumerate(
            zip(current_footer, next_footer),
            start=1
        ):

            similarity_records.append({
                "page_from": page_number,
                "page_to": page_number + 1,
                "text1": text1,
                "text2": text2,
                "region": "footer",
                "line": line_number,
                "score": fuzz.ratio(text1, text2)
            })

    return pd.DataFrame(similarity_records)


# def identify_boilerplate(
#     similarity_df,
#     mean_threshold=90,
#     std_threshold=10
# ):
#     """
#     Identify repeated boilerplate lines using similarity statistics.

#     Parameters
#     ----------
#     similarity_df : pandas.DataFrame
#         DataFrame containing:
#             region
#             line
#             score

#     mean_threshold : float
#         Minimum average similarity required.

#     std_threshold : float
#         Maximum allowed standard deviation.

#     Returns
#     -------
#     pandas.DataFrame
#         Summary statistics and classification.
#     """

#     statistics = (
#         similarity_df
#         .groupby(["region", "line"])["score"]
#         .agg(
#             mean_score="mean",
#             std_score="std"
#         )
#         .reset_index()
#     )

#     statistics["is_boilerplate"] = (
#         (statistics["mean_score"] >= mean_threshold)
#         &
#         (statistics["std_score"] <= std_threshold)
#     )

#     return statistics


# =========================
# 6. Statistical Boilerplate Detection
# =========================

def identify_boilerplate(
    similarity_df,
    mean_threshold=90,
    std_threshold=10
):
    """
    Identify repeated boilerplate lines using similarity statistics.

    Parameters
    ----------
    similarity_df : pandas.DataFrame
        DataFrame containing:
            region
            line
            score

    mean_threshold : float
        Minimum average similarity required.

    std_threshold : float
        Maximum allowed standard deviation.

    Returns
    -------
    pandas.DataFrame
        Summary statistics and classification.
    """

    statistics = (
        similarity_df
        .groupby(["region", "line"])["score"]
        .agg(
            mean_score="mean",
            std_score="std"
        )
        .reset_index()
    )

    statistics["is_boilerplate"] = (
        (statistics["mean_score"] >= mean_threshold)
        &
        (statistics["std_score"] <= std_threshold)
    )

    return statistics


# def remove_boilerplate(pdf, boilerplate_config):
#     cleaned_pdf = []

#     for page in pdf:

#         # Keep only non-empty lines
#         page_lines = [
#             line.strip()
#             for line in page.splitlines()
#             if line.strip()
#         ]

#         # Identify lines to remove
#         lines_to_remove = set()

#         for _, row in boilerplate_config.iterrows():

#             region = row["region"]
#             line_number = row["line"]

#             if region == "header":

#                 index = line_number - 1

#             elif region == "footer":

#                 index = len(page_lines) - line_number

#             else:
#                 continue

#             # Make sure the index exists on this page
#             if 0 <= index < len(page_lines):
#                 lines_to_remove.add(index)

#         # Keep all lines except detected boilerplate
#         cleaned_page_lines = [
#             line
#             for index, line in enumerate(page_lines)
#             if index not in lines_to_remove
#         ]

#         cleaned_page = "\n".join(cleaned_page_lines)

#         cleaned_pdf.append(cleaned_page)

#     return cleaned_pdf


# =========================
# 9. Remove Boilerplate
# =========================

def remove_boilerplate(
    pdf,
    boilerplate_config
):

    cleaned_pdf = []

    boilerplate_positions = {
        (row["region"], row["line"])
        for _, row in boilerplate_config.iterrows()
    }

    for page in pdf:

        page_lines = get_page_lines(page)

        lines_to_remove = set()

        for region, line_number in boilerplate_positions:

            if region == "header":

                index = line_number - 1

            elif region == "footer":

                index = len(page_lines) - line_number

            else:

                continue

            if 0 <= index < len(page_lines):

                lines_to_remove.add(index)

        cleaned_page = "\n".join(
            line
            for index, line in enumerate(page_lines)
            if index not in lines_to_remove
        )

        cleaned_pdf.append(cleaned_page)

    return cleaned_pdf

### Text Preprocessing

##### Cell 1: Regex Based Preprocessing

In [4]:
# Task 1: Normalize the text as much as possible
normalized_pdf = []
i = 0

for page in pdf:
    normalized_page = normalize_text(page)
    normalized_pdf.append(normalized_page)    


# Comparing The cleaned PDF vs Original PDF
for i in range(0,5):
    print("---------------------------------------")
    print("------------- Original PDF ------------") 
    print(pdf[i])
    print("---------------------------------------\n")

    print("---------------------------------------")
    print("------------- Cleaned PDF ------------") 
    print(normalized_pdf[i])
    print("---------------------------------------\n")

---------------------------------------
------------- Original PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
 
---------------------------------------

---------------------------------------
------------- Cleaned PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quant

#### Cell 2: Statistics Based Preprocessing

##### Drawing statistics to find the Pattern

In [27]:
similar_line_pdf = compute_line_similarity(pdf, 3)

similar_line_pdf

,page_from,page_to,text1,text2,region,line,score
0,1,2,Basics of Mathematics,Basics of Mathematics ...,header,1,26.415094
1,1,2,Sr. No. Topic name Page number,Section A,header,2,20.512821
2,1,2,1. Qualitative and Quantitative Analysis 2,A.1 Introduction to Qualitative and,header,3,49.350649
3,1,2,19. Series 205,experience (UX),footer,1,34.482759
4,1,2,20. Problems on Ages 210,Researching competitors’ website pages to iden...,footer,2,24.347826
...,...,...,...,...,...,...,...
473,221,222,Basics of Mathematics ...,Basics of Mathematics ...,footer,1,99.270073
474,222,223,Basics of Mathematics ...,Basics of Mathematics ...,header,1,99.270073
475,222,223,Basics of Mathematics ...,Basics of Mathematics ...,footer,1,99.270073
476,223,224,Basics of Mathematics ...,Basics of Mathematics ...,header,1,99.270073


In [6]:
# suspect_similarity_dataset = pd.DataFrame(columns=['text1','text2', 'line', 'page_from', 'page_to', 'region',  'score'])

# for page in similar_line_pdf: 
#     for header in page['header_scores']:
#         new_row = {
#             'page_from': page['pages'][0],
#             'page_to': page['pages'][1],
#             'region': 'header',
#             'line': header['line'],
#             'text1': header['text1'],
#             'text2': header['text2'],
#             'score': header['score']
#         }
#         suspect_similarity_dataset.loc[len(suspect_similarity_dataset)] = new_row
        
#     for footer in page['footer_scores']:
#         new_row = {
#             'page_from': page['pages'][0],
#             'page_to': page['pages'][1],
#             'region': 'footer',
#             'line': footer['line'],
#             'text1': footer['text1'],
#             'text2': footer['text2'],
#             'score': footer['score']
#         }
#         suspect_similarity_dataset.loc[len(suspect_similarity_dataset)] = new_row


# suspect_similarity_dataset.head(10)


In [7]:
# """
# Computing the mean and the Std
# """

# # Headers
# suspect_similarity_mean = suspect_similarity_dataset.groupby(['region', 'line'])['score'].mean()
# suspect_similarity_std = suspect_similarity_dataset.groupby(['region', 'line'])['score'].std()


# print("::::::::::::::: Line Statistics :::::::::::::::")
# print("Mean :::>")
# print(suspect_similarity_mean)

# print("\n\n")
# print("Std :::>")
# print(suspect_similarity_std)

In [28]:
# data = identify_boilerplate(similarity_df=suspect_similarity_dataset)
statistics_df = identify_boilerplate(similarity_df=similar_line_pdf)


# boilerplate_df = data[data['is_boilerplate'] == True]

print(statistics_df)


   region  line  mean_score  std_score  is_boilerplate
0  footer     1   95.856193  15.738222           False
1  footer     2   35.873401  21.741000           False
2  footer     3   28.987282  10.501335           False
3  header     1   98.873913   4.879473            True
4  header     2   25.551731   8.896255           False
5  header     3   30.893231  13.263025           False


In [29]:
# =========================
# 8. Extract Boilerplate Configuration
# =========================

boilerplate_config = (
    statistics_df[
        statistics_df["is_boilerplate"]
    ][["region", "line"]]
)

boilerplate_config


# =========================
# 10. Clean PDF
# =========================

cleaned_pdf = remove_boilerplate(
    pdf=pdf,
    boilerplate_config=boilerplate_config
)

In [32]:
cleaned_pdf[2]

"Interpreting customer interviews and survey results to determine user preferences, and\nsetting the direction for new product or feature developments\n2. Thematic analysis\nThematic analysis helps to identify, analyze, and interpret patterns in qualitative data, and can\nbe done with tools like Dovetail and Thematic.\nWhile content analysis and thematic analysis seem similar, they're different in concept:\nContent analysis can be applied to both qualitative and quantitative data, and focuses on\nidentifying frequencies and recurring words and subjects.\nThematic analysis can only be applied to qualitative data, and focuses on identifying patterns\nand ‘themes’.\nHow thematic analysis can help your team\nThematic analysis can be used by pretty much anyone: from product marketers, to customer\nrelationship managers, to UX researchers.\nFor example, product teams can use thematic analysis to better understand user behaviors\nand needs, and to improve UX. By analyzing customer feedback, y

In [11]:
text = "".join(cleaned_pdf)

text

"Sr. No. Topic name Page number\n1. Qualitative and Quantitative Analysis 2\n2. Number and Number System, problems on Numbers 7\n3. Fractions, Decimals and Numbers 32\n4. Divisibility and Divisibility Test 39\n5. HCF and LCM 44\n6. Squares, Cubes, Square-roots & Cube-roots 51\n7. Simplifications 58\n8. Ratio and Proportion 69\n9. Chain rule\n74\n10. Averages 82\n11. Percentages 91\n12. Profit and Loss 101\n13. Simple and Compound Interest 110\n14. Heights, Distances & Basic Trigonometry 135\n15. Time, Work and Wages 142\n16. Time and Distances 162\n17. Trains, Boats and Streams 176\n18. Clock 197\n19. Series 205\n20. Problems on Ages   210\n21. Discounts – its types and Analysis 216Section A\nA.1 Introduction to Qualitative and\nQuantitative Analysis\nQualitative Data Analysis\nQualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting\nqualitative data—non-numeric, conceptual information and user feedback—to capture\nthemes and patterns, answer research 

### Chunking the PDF
:-- Using recursive chunking

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        " ",
        ".",
        ",",
        "\u200b",  # Zero-width space
        "\uff0c",  # Fullwidth comma
        "\u3001",  # Ideographic comma
        "\uff0e",  # Fullwidth full stop
        "\u3002",  # Ideographic full stop
        "",
    ],
)

all_splits = text_splitter.split_text(text)

print(f"Split PDF text into {len(all_splits)} sub-documents.")


Split PDF text into 22 sub-documents.


In [13]:
for i, chunk in enumerate(all_splits):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\n--- First half of chunk {i+1} ---")
    print(chunk[:300])
    print(f"\n--- Second half of chunk {i+1} ---")
    print(chunk[300:])
    print(f"\nLength: {len(chunk)}")

    if i == 2:
        break


--- Chunk 1 ---

--- First half of chunk 1 ---
Sr. No. Topic name Page number
1. Qualitative and Quantitative Analysis 2
2. Number and Number System, problems on Numbers 7
3. Fractions, Decimals and Numbers 32
4. Divisibility and Divisibility Test 39
5. HCF and LCM 44
6. Squares, Cubes, Square-roots & Cube-roots 51
7. Simplifications 58
8. Ratio

--- Second half of chunk 1 ---
 and Proportion 69
9. Chain rule
74
10. Averages 82
11. Percentages 91
12. Profit and Loss 101
13. Simple and Compound Interest 110
14. Heights, Distances & Basic Trigonometry 135
15. Time, Work and Wages 142
16. Time and Distances 162
17. Trains, Boats and Streams 176
18. Clock 197
19. Series 205
20. Problems on Ages   210
21. Discounts – its types and Analysis 216Section A
A.1 Introduction to Qualitative and
Quantitative Analysis
Qualitative Data Analysis
Qualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting
qualitative data—non-numeric, conceptual information and user feedb

### Embeddings

In [14]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(all_splits)

print(f"Each embedding has {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions


In [15]:
similarities = cosine_similarity(embeddings)
print("Similarity matrix:")
print(similarities)
# You'll see sentences 1, 2, and 4 are more similar to each other


Similarity matrix:
[[ 1.          0.60902035  0.46741772  0.41765928  0.31461573  0.39369875
   0.34920907  0.48842028  0.559707    0.50960577  0.61037403  0.46877468
   0.2865466   0.28245527  0.25410283  0.30219734  0.2787121   0.2485761
   0.22322056  0.3150742   0.3674897   0.30143046]
 [ 0.60902035  0.9999999   0.8082719   0.70771253  0.5715337   0.6755384
   0.6358664   0.75199354  0.54995906  0.6357827   0.66227096  0.29256487
   0.11715031  0.10368641  0.04718842  0.0780332   0.1097618   0.07927002
   0.07696318  0.13564862  0.14791796  0.13884303]
 [ 0.46741772  0.8082719   1.          0.852163    0.59900427  0.65352976
   0.584928    0.70955765  0.52519727  0.63953185  0.63214946  0.26657555
   0.10993231  0.10303891  0.04962326  0.08211279  0.10259019  0.06191434
   0.07400539  0.10947983  0.1051769   0.13139331]
 [ 0.41765928  0.70771253  0.852163    1.          0.7548932   0.7117351
   0.6359838   0.6850221   0.45739776  0.56506455  0.578096    0.20360826
   0.07673024  0.

## Storing into the Database

In [16]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [17]:
print(index.ntotal)

22


In [18]:
# query = "What is qualitative data analysis?"
query = "Steps to effective quantitative data analysis "

query_embedding = model.encode([query])


## Searching into Database

In [19]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

In [20]:
retrived_text = []
for idx in indices[0]:
    retrived_text.append(all_splits[idx])

retrived_text

['experiences.\n4. Enhanced User Satisfaction:  Pinpoints strengths and weaknesses, enabling improvements that\nboost conversions and delight users.\nSteps to effective quantitative data analysis\nQuantitative Data Analysis Steps:\n1. Collect Data:  Use surveys, website analytics, heatmaps, or session recordings, focusing on\nclose-ended questions for numerical data.\n2. Clean Data:  Remove errors, duplicates, omissions, and outliers to ensure accuracy and\nconsistency.\n3. Analyze & Interpret:  Use descriptive analysis (e.g., mean, median) to summarize data or\ninferential analysis (e.g., t-tests) to find relationships and make predictions. Identify actionable\ninsights.\n4. Visualize & Share:  Present findings with charts, graphs, or dashboards to highlight patterns\nand relationships.\nDifference between Quantitative data analysis and Qualitative data analysisSection BDivisibility Rule of 1\nDivisibility Rules',
 'Quantitative data analysis is the process of analyzing and interpreti

## Response Generation using LLM
---

### Cell 1
#### Responsibility : - Configure LLM Model

In [21]:
# Client Configuration
API_KEY = 'AQ.Ab8RN6LdZl8pxdRhnCTAMsjCvQuzFv1CRJt3JGnqdkmhUR-GhQ'

client = genai.Client(api_key=API_KEY)

### Cell 2
#### Responsibility : - Prompt Builder

In [22]:
context = "\n\n".join(retrived_text)

prompt = f"""
You are an AI assistant.

Answer ONLY using the provided context.

If the answer is not available,
say "I don't know."

Context:
{context}

Query:
{query}
"""

In [23]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents= prompt
)

In [24]:
print("response")
print(response.text)

response
Steps to effective quantitative data analysis are:
1.  **Collect Data:** Use surveys, website analytics, heatmaps, or session recordings, focusing on close-ended questions for numerical data.
2.  **Clean Data:** Remove errors, duplicates, omissions, and outliers to ensure accuracy and consistency.
3.  **Analyze & Interpret:** Use descriptive analysis (e.g., mean, median) to summarize data or inferential analysis (e.g., t-tests) to find relationships and make predictions. Identify actionable insights.
4.  **Visualize & Share:** Present findings with charts, graphs, or dashboards to highlight patterns and relationships.
